# Homework 4: Pretrained Transformers

---

**Student Name:** Timothy Pokanai

**Student ID:** 200004735

**CodaBench Username:** tim-pokanai

**Assigned Task (from Group Assignments A4.txt):** Group *17* - CodaBench URL: *https://www.codabench.org/competitions/14955/*

**Self-Chosen Secondary Task:** Group *26* - CodaBench URL: *https://www.codabench.org/competitions/15005/*

---

### AI Tool Usage Declaration

*Copilot was used to help complete the zero-shot best prompt testing section. Since my assigned task didn't have test data that was also prelabeled, I used Copilot to help create a validation cycle as well to calculate evaluation metrics that I assume also apply to the test cycle, since here we don't fine-tune the zero-shot models.*

---
## 0. Setup & Dependencies

Install and import all libraries you need here.

In [7]:
%pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.5 MB/s eta 0:00:00


In [8]:
# Install required packages (uncomment as needed)
# !pip install transformers datasets scikit-learn torch evaluate accelerate
# !pip install sentencepiece  # needed for some tokenizers

# Core imports
import random
import numpy as np
import pandas as pd

# HuggingFace
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [1]:
# I used Google Colab for their T4 GPUs, to access my datasets I needed to move them to my drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
# PART 1 — Assigned Task

Complete **all** steps (#3.1 Fine-tuning, #3.2 Zero-shot, #3.3 Baselines) for your **assigned** task.

## 1.1 Dataset Loading & Exploration

In [ ]:
from pathlib import Path

# I'm using the team's provided datasets instead of pulling directly
# from hugging face since they cleaned up the data nicely
DATA_DIR = Path("/content/drive/MyDrive/public_data/")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test_public.csv")

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

Train shape: (90000, 6)
Test shape:  (10000, 5)


In [ ]:
print("=== Train ===")
print(train_df.dtypes)
print("\nMissing values:\n", train_df.isnull().sum())

# Test dataset has no labels
print("\n=== Test ===")
print(test_df.dtypes)
print("\nMissing values:\n", test_df.isnull().sum())

print("\n=== Label distribution (train) ===")
print(train_df["label"].value_counts().sort_index())

print("\n=== Sample rows (train) ===")
train_df.head()

=== Train ===
id           int64
title       object
year         int64
synopsis    object
reviews     object
label        int64
dtype: object

Missing values:
 id            0
title         0
year          0
synopsis    190
reviews       0
label         0
dtype: int64

=== Test ===
id           int64
title       object
year         int64
synopsis    object
reviews     object
dtype: object

Missing values:
 id           0
title        0
year         0
synopsis    14
reviews      0
dtype: int64

=== Label distribution (train) ===
label
1       96
2     5705
3    58034
4    26093
5       72
Name: count, dtype: int64

=== Sample rows (train) ===


,id,title,year,synopsis,reviews,label
0,18992,The Secrets of Jonathan Sperry,2008,"The lives of twelve year old buddies Dustin, A...",most of the faith based movies (akaCHRISTPLOIT...,3
1,63023,Retirement Plan,2024,"At apathy with his life, Ray dreams of the bea...",almost forgot this was the whole point,4
2,82745,Love Jones,1997,Darius Lovehall is a young black poet in Chica...,“Romance is about the possibility of the thing...,4
3,90403,How I Learned to Fly,2023,Two African-American teenage boys are suddenly...,An intimate character study with stellar perfo...,3
4,47720,Tumbleweeds,1999,Whenever trouble strikes in one of her relatio...,Feel good mother and daughter road trip movie....,3


---
## 1.2 Fine-tuning Pretrained Models (#3.1)

Select **two** different pretrained transformer models (e.g. BERT, RoBERTa, DistilBERT, DeBERTa, ModernBERT, ELECTRA, T5, …).  
The models must **not** already be fine-tuned on your specific task.

### Fine-tuned Model 1

In [7]:
MODEL_1_NAME = "distilroberta-base"

num_labels = train_df["label"].nunique()
label_ids = sorted(train_df["label"].unique())
id2label = {i: str(lbl) for i, lbl in enumerate(label_ids)}
label2id = {str(lbl): i for i, lbl in enumerate(label_ids)}

tokenizer_1 = AutoTokenizer.from_pretrained(MODEL_1_NAME)
model_1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_1_NAME,
    num_labels=num_labels,
)

model_1.config.id2label = id2label
model_1.config.label2id = label2id

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

TEXT_COLS = ["title", "synopsis", "reviews"]

# Here I'm joining each column's text into one column, done row-wise
def build_input_text(row):
    parts = []
    for col in TEXT_COLS:
        val = row[col]
        if pd.notna(val):
            text = str(val).strip()
            if text:
                parts.append(text)
    return " </s> ".join(parts)

train_df_m1 = train_df.copy()
test_df_m1 = test_df.copy()

train_df_m1["text"] = train_df_m1.apply(build_input_text, axis=1)
test_df_m1["text"] = test_df_m1.apply(build_input_text, axis=1)

train_split_df, val_split_df = train_test_split(
    train_df_m1[["text", "label"]],
    test_size=0.1,
    random_state=SEED,
    stratify=train_df_m1["label"],
)

raw_ds_1 = DatasetDict(
    {
        "train": Dataset.from_pandas(train_split_df.reset_index(drop=True), preserve_index=False),
        "validation": Dataset.from_pandas(val_split_df.reset_index(drop=True), preserve_index=False),
        "test": Dataset.from_pandas(test_df_m1[["text"]].reset_index(drop=True), preserve_index=False),
    }
)

def tokenize_batch_model_1(batch):
    return tokenizer_1(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized_ds_1 = raw_ds_1.map(tokenize_batch_model_1, batched=True)

tokenized_train_1 = tokenized_ds_1["train"].remove_columns(["text"])
tokenized_val_1 = tokenized_ds_1["validation"].remove_columns(["text"])
tokenized_test_1 = tokenized_ds_1["test"].remove_columns(["text"])

print(tokenized_train_1)
print(tokenized_val_1)
print(tokenized_test_1)

Map:   0%|          | 0/81000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 81000
})
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 9000
})
Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10000
})


In [12]:
from transformers import DataCollatorWithPadding

data_collator_1 = DataCollatorWithPadding(tokenizer=tokenizer_1)

def compute_metrics_model_1(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

def add_trainer_labels(batch):
    return {"labels": [label2id[str(int(lbl))] for lbl in batch["label"]]}

train_for_trainer_1 = tokenized_train_1.map(add_trainer_labels, batched=True)
val_for_trainer_1 = tokenized_val_1.map(add_trainer_labels, batched=True)

train_for_trainer_1 = train_for_trainer_1.remove_columns(["label"])
val_for_trainer_1 = val_for_trainer_1.remove_columns(["label"])

# Apply torch format after label remapping to avoid any format/type issues.
train_for_trainer_1.set_format("torch")
val_for_trainer_1.set_format("torch")
tokenized_test_1.set_format("torch")

training_args_1 = TrainingArguments(
    output_dir="./outputs/model_1_distilroberta",
    num_train_epochs=1,
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    seed=SEED,
)

trainer_1 = Trainer(
    model=model_1,
    args=training_args_1,
    train_dataset=train_for_trainer_1,
    eval_dataset=val_for_trainer_1,
    data_collator=data_collator_1,
    compute_metrics=compute_metrics_model_1,
)

train_result_1 = trainer_1.train()
print("Selected learning_rate:", training_args_1.learning_rate)
print("Selected weight_decay:", training_args_1.weight_decay)
print("Train loss:", train_result_1.training_loss)

Map:   0%|          | 0/81000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.856942,0.831010,0.644778,0.505525


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Selected learning_rate: 0.0002
Selected weight_decay: 0.05
Train loss: 0.8408085144769291


In [13]:
# Recall that test split contains data that was not prelabelled, unlike train/validate
test_predictions_1 = trainer_1.predict(tokenized_test_1)
test_pred_class_ids_1 = np.argmax(test_predictions_1.predictions, axis=-1)

predicted_labels_1 = [int(id2label[idx]) for idx in test_pred_class_ids_1]

model_1_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": predicted_labels_1,
    }
)

print(model_1_test_results.head())
print("Total predictions:", len(model_1_test_results))
print("Label distribution:\n", model_1_test_results["predicted_label"].value_counts().sort_index())

model_1_test_results.to_csv("model_1_test_predictions.csv", index=False)
print("Saved: model_1_test_predictions.csv")

      id  predicted_label
0  74233                3
1  28972                3
2  31376                3
3  18117                3
4  72814                3
Total predictions: 10000
Label distribution:
 predicted_label
3    10000
Name: count, dtype: int64
Saved: model_1_test_predictions.csv


### Fine-tuned Model 2

In [14]:
MODEL_2_NAME = "prajjwal1/bert-tiny"

tokenizer_2 = AutoTokenizer.from_pretrained(MODEL_2_NAME)
model_2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_2_NAME,
    num_labels=num_labels,
)

model_2.config.id2label = id2label
model_2.config.label2id = label2id

print("Loaded:", MODEL_2_NAME)
print("Number of classes:", num_labels)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

Loaded: prajjwal1/bert-tiny
Number of classes: 5


In [15]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

TEXT_COLS_2 = ["title", "synopsis", "reviews"]

def build_input_text_model_2(row):
    parts = []
    for col in TEXT_COLS_2:
        val = row[col]
        if pd.notna(val):
            text = str(val).strip()
            if text:
                parts.append(text)
    return " </s> ".join(parts)

train_df_m2 = train_df.copy()
test_df_m2 = test_df.copy()

train_df_m2["text"] = train_df_m2.apply(build_input_text_model_2, axis=1)
test_df_m2["text"] = test_df_m2.apply(build_input_text_model_2, axis=1)

train_split_df_m2, val_split_df_m2 = train_test_split(
    train_df_m2[["text", "label"]],
    test_size=0.1,
    random_state=SEED,
    stratify=train_df_m2["label"],
)

raw_ds_2 = DatasetDict(
    {
        "train": Dataset.from_pandas(train_split_df_m2.reset_index(drop=True), preserve_index=False),
        "validation": Dataset.from_pandas(val_split_df_m2.reset_index(drop=True), preserve_index=False),
        "test": Dataset.from_pandas(test_df_m2[["text"]].reset_index(drop=True), preserve_index=False),
    }
)

def tokenize_batch_model_2(batch):
    return tokenizer_2(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized_ds_2 = raw_ds_2.map(tokenize_batch_model_2, batched=True)

tokenized_train_2 = tokenized_ds_2["train"].remove_columns(["text"])
tokenized_val_2 = tokenized_ds_2["validation"].remove_columns(["text"])
tokenized_test_2 = tokenized_ds_2["test"].remove_columns(["text"])

print(tokenized_train_2)
print(tokenized_val_2)
print(tokenized_test_2)

Map:   0%|          | 0/81000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 81000
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 9000
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10000
})


In [16]:
from transformers import DataCollatorWithPadding

data_collator_2 = DataCollatorWithPadding(tokenizer=tokenizer_2)

def compute_metrics_model_2(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
    }

def add_trainer_labels_model_2(batch):
    return {"labels": [label2id[str(int(lbl))] for lbl in batch["label"]]}

train_for_trainer_2 = tokenized_train_2.map(add_trainer_labels_model_2, batched=True)
val_for_trainer_2 = tokenized_val_2.map(add_trainer_labels_model_2, batched=True)

train_for_trainer_2 = train_for_trainer_2.remove_columns(["label"])
val_for_trainer_2 = val_for_trainer_2.remove_columns(["label"])

train_for_trainer_2.set_format("torch")
val_for_trainer_2.set_format("torch")
tokenized_test_2.set_format("torch")

training_args_2 = TrainingArguments(
    output_dir="./outputs/model_2_berttiny",
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    seed=SEED,
)

trainer_2 = Trainer(
    model=model_2,
    args=training_args_2,
    train_dataset=train_for_trainer_2,
    eval_dataset=val_for_trainer_2,
    data_collator=data_collator_2,
    compute_metrics=compute_metrics_model_2,
)

train_result_2 = trainer_2.train()
print("Selected learning_rate:", training_args_2.learning_rate)
print("Selected weight_decay:", training_args_2.weight_decay)
print("Train loss:", train_result_2.training_loss)

Map:   0%|          | 0/81000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.750822,0.715585,0.686778,0.637471
2,0.591925,0.683244,0.711333,0.680623
3,0.691722,0.675801,0.719222,0.697902


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.en

Selected learning_rate: 3e-05
Selected weight_decay: 0.01
Train loss: 0.6960550662382149


In [17]:
test_predictions_2 = trainer_2.predict(tokenized_test_2)
test_pred_class_ids_2 = np.argmax(test_predictions_2.predictions, axis=-1)

predicted_labels_2 = [int(id2label[idx]) for idx in test_pred_class_ids_2]

model_2_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": predicted_labels_2,
    }
)

print(model_2_test_results.head())
print("Total predictions:", len(model_2_test_results))
print("Label distribution:\n", model_2_test_results["predicted_label"].value_counts().sort_index())

model_2_test_results.to_csv("model_2_test_predictions.csv", index=False)
print("Saved: model_2_test_predictions.csv")

      id  predicted_label
0  74233                3
1  28972                3
2  31376                3
3  18117                3
4  72814                3
Total predictions: 10000
Label distribution:
 predicted_label
2     133
3    7450
4    2417
Name: count, dtype: int64
Saved: model_2_test_predictions.csv


### Discussion — Fine-tuned Models (10%)

Answer all of the following in this cell:

- Describe the models you selected and what steps you needed to perform to fine-tune them for your task.

- List the size of each model in number of parameters and discuss the dataset that the model was pretrained on, as well as the compute requirements used for pretraining (if details are available, you may need to do some searching to find the answer), e.g., 10 Tesla RTX 8000 GPUs were trained for 24 hours

*For fine-tuning, I selected two pretrained transformer models with different sizes: DistilRoBERTa (distilroberta-base) and BERT-tiny (prajjwal1/bert-tiny). I chose these to compare a medium-sized encoder against a lightweight baseline and see how model capacity affects performance on the same task.*

*Fine-tuning each model followed the same pipeline. I first loaded the train and test datasets, then merged the relevant text fields into one input string per data point. After that, I tokenized all inputs with each model’s matching tokenizer and converted labels into class IDs using label mappings. I used Hugging Face Trainer with DataCollatorWithPadding, defined evaluation metrics (accuracy and weighted F1), and set both model's respective training arguments. After training, I evaluated on the validation split and then generated predictions for the unlabeled test split, converting predicted class IDs back into original label values for Codabench submission files.*

*DistilRoBERTa (distilroberta-base) has about 82 million parameters, while BERT-tiny (prajjwal1/bert-tiny) has about 4.4 million parameters. DistilRoBERTa is a distilled version of RoBERTa-base, so it inherits RoBERTa’s training data setup which consists of roughly 160GB total text of large English corpora. RoBERTa’s original pretraining compute is documented at around 1024 V100 GPUs for about 1 day for the main run, but exact distillation-time compute for distilroberta-base is not clearly reported. BERT-tiny is a compact BERT architecture checkpoint, commonly containing 2 layers, a hidden size of 128, and 2 attention heads. The underlying BERT pretraining corpus is about 3.3 billion words. Original BERT (Bert-base/large) pretraining used Google Cloud TPUs, but the hardware details for this specific model checkpoint were not specified.*

---
## 1.3 Zero-shot Classification (#3.2)

Use **two** different pretrained language models in a zero-shot setting (no additional training).  
Try multiple prompts on a small portion of the **training** data to identify the best prompt, then evaluate on the **test** set using only that prompt.

If you are using Llama.cpp, include the file in your submission and enter your filename here: *(filename)*

And include instructions on how to run the code: *(instructions)*

Otherwise, complete the following cells.

### Zero-shot Model 1 — Prompt Engineering

In [18]:
from transformers import pipeline

ZS_MODEL_1_NAME = "facebook/bart-large-mnli"

zs_classifier_1 = pipeline(
    task="zero-shot-classification",
    model=ZS_MODEL_1_NAME,
    tokenizer=ZS_MODEL_1_NAME,
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
if len(label_ids) == 5:
    candidate_labels_zs1 = ["very negative", "negative", "neutral", "positive", "very positive"]
    label_text_to_id_zs1 = {
        "very negative": label_ids[0],
        "negative": label_ids[1],
        "neutral": label_ids[2],
        "positive": label_ids[3],
        "very positive": label_ids[4],
    }
else:
    candidate_labels_zs1 = [str(lbl) for lbl in label_ids]
    label_text_to_id_zs1 = {str(lbl): lbl for lbl in label_ids}

prompt_variants = [
    {
        "name": "sentiment-direct",
        "hypothesis_template": "This movie review is {}.",
    },
    {
        "name": "rating-tone",
        "hypothesis_template": "The overall audience sentiment is {}.",
    },
    {
        "name": "critic-stance",
        "hypothesis_template": "The critic's stance toward this movie is {}.",
    },
]

# Use a small sample for quick prompt comparison
zs1_eval_sample = train_df.sample(n=min(250, len(train_df)), random_state=SEED)

variant_results_zs1 = []

for variant in prompt_variants:
    preds = []

    for text in zs1_eval_sample["reviews"].fillna("").astype(str).tolist():
        out = zs_classifier_1(
            text,
            candidate_labels=candidate_labels_zs1,
            hypothesis_template=variant["hypothesis_template"],
            multi_label=False,
        )
        top_label_text = out["labels"][0]
        preds.append(label_text_to_id_zs1[top_label_text])

    y_true = zs1_eval_sample["label"].tolist()
    y_pred = preds

    variant_results_zs1.append(
        {
            "variant": variant["name"],
            "hypothesis_template": variant["hypothesis_template"],
            "accuracy": accuracy_score(y_true, y_pred),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        }
    )

zs1_prompt_results_df = pd.DataFrame(variant_results_zs1).sort_values(
    by=["f1_weighted", "accuracy"], ascending=False
)

print("Prompt variant comparison (Model 1, sample):")
print(zs1_prompt_results_df)

BEST_PROMPT_ZS1 = zs1_prompt_results_df.iloc[0]["hypothesis_template"]
print("\nSelected BEST_PROMPT_ZS1:", BEST_PROMPT_ZS1)

Prompt variant comparison (Model 1, sample):
            variant                           hypothesis_template  accuracy  \
1       rating-tone         The overall audience sentiment is {}.     0.324   
0  sentiment-direct                      This movie review is {}.     0.276   
2     critic-stance  The critic's stance toward this movie is {}.     0.256   

   f1_weighted  
1     0.360665  
0     0.230867  
2     0.198138  

Selected BEST_PROMPT_ZS1: The overall audience sentiment is {}.


In [23]:
if "BEST_PROMPT_ZS1" not in globals() or not BEST_PROMPT_ZS1:
    BEST_PROMPT_ZS1 = "The overall audience sentiment is {}."

# Ensure candidate labels/mapping exist even if the previous cell wasn't run.
if "candidate_labels_zs1" not in globals() or "label_text_to_id_zs1" not in globals():
    if len(label_ids) == 5:
        candidate_labels_zs1 = ["very negative", "negative", "neutral", "positive", "very positive"]
        label_text_to_id_zs1 = {
            "very negative": label_ids[0],
            "negative": label_ids[1],
            "neutral": label_ids[2],
            "positive": label_ids[3],
            "very positive": label_ids[4],
        }
    else:
        candidate_labels_zs1 = [str(lbl) for lbl in label_ids]
        label_text_to_id_zs1 = {str(lbl): lbl for lbl in label_ids}

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

def predict_zs1_labels(texts, batch_size=16, desc="Predicting"):
    texts = ["" if pd.isna(t) else str(t) for t in texts]
    preds = []
    iterator = range(0, len(texts), batch_size)
    if tqdm is not None:
        iterator = tqdm(iterator, total=(len(texts) + batch_size - 1) // batch_size, desc=desc)

    for i in iterator:
        batch_texts = texts[i : i + batch_size]
        outputs = zs_classifier_1(
            batch_texts,
            candidate_labels=candidate_labels_zs1,
            hypothesis_template=BEST_PROMPT_ZS1,
            multi_label=False,
            truncation=True,
        )

        if isinstance(outputs, dict):
            outputs = [outputs]

        for out in outputs:
            top_label_text = out["labels"][0]
            preds.append(label_text_to_id_zs1[top_label_text])

    return preds

# Take a validation split from train, and run it like a test split using prelabelled values for metrics
if "val_split_df" in globals() and "text" in val_split_df.columns and "label" in val_split_df.columns:
    zs1_val_df = val_split_df.copy().reset_index(drop=True)
else:
    zs1_val_df = train_df.sample(n=min(1000, len(train_df)), random_state=SEED).copy().reset_index(drop=True)
    zs1_val_df["text"] = zs1_val_df["reviews"].fillna("").astype(str)

zs1_val_preds = predict_zs1_labels(
    zs1_val_df["text"].tolist(),
    batch_size=16,
    desc="Validation zero-shot",
)

zs1_val_true = zs1_val_df["label"].tolist()
zs1_val_acc = accuracy_score(zs1_val_true, zs1_val_preds)
zs1_val_f1w = f1_score(zs1_val_true, zs1_val_preds, average="weighted")
zs1_val_f1m = f1_score(zs1_val_true, zs1_val_preds, average="macro")

print("Zero-shot Model 1 metrics (validation):")
print(f"Accuracy: {zs1_val_acc:.4f}")
print(f"F1 (Weighted): {zs1_val_f1w:.4f}")
print(f"F1 (Macro): {zs1_val_f1m:.4f}")

# Now run the actual unlabelled test split
test_texts_zs1 = test_df["reviews"].fillna("").astype(str).tolist()
zs1_test_preds = predict_zs1_labels(
    test_texts_zs1,
    batch_size=16,
    desc="Test zero-shot",
)

zs1_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": zs1_test_preds,
    }
)

print("\nZero-shot Model 1 test prediction preview:")
print(zs1_test_results.head())
print("Total predictions:", len(zs1_test_results))
print("Label distribution:\n", zs1_test_results["predicted_label"].value_counts().sort_index())

zs1_test_results.to_csv("zs_model_1_test_predictions.csv", index=False)
print("Saved: zs_model_1_test_predictions.csv")

Validation zero-shot:   0%|          | 0/563 [00:00<?, ?it/s]

Zero-shot Model 1 metrics (validation):
Accuracy: 0.3436
F1 (Weighted): 0.3795
F1 (Macro): 0.1945


Test zero-shot:   0%|          | 0/625 [00:00<?, ?it/s]


Zero-shot Model 1 test prediction preview:
      id  predicted_label
0  74233                3
1  28972                2
2  31376                2
3  18117                2
4  72814                4
Total predictions: 10000
Label distribution:
 predicted_label
1     153
2    4080
3    2036
4    3594
5     137
Name: count, dtype: int64
Saved: zs_model_1_test_predictions.csv


### Zero-shot Model 2 — Prompt Engineering

In [24]:
ZS_MODEL_2_NAME = "typeform/distilbert-base-uncased-mnli"

zs_classifier_2 = pipeline(
    task="zero-shot-classification",
    model=ZS_MODEL_2_NAME,
    tokenizer=ZS_MODEL_2_NAME,
)

config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [28]:
if len(label_ids) == 5:
    candidate_labels_zs2 = ["very negative", "negative", "neutral", "positive", "very positive"]
    label_text_to_id_zs2 = {
        "very negative": label_ids[0],
        "negative": label_ids[1],
        "neutral": label_ids[2],
        "positive": label_ids[3],
        "very positive": label_ids[4],
    }
else:
    candidate_labels_zs2 = [str(lbl) for lbl in label_ids]
    label_text_to_id_zs2 = {str(lbl): lbl for lbl in label_ids}

prompt_variants_zs2 = [
    {
        "name": "public-reception",
        "hypothesis_template": "The public reception of this movie would be {}.",
    },
    {
        "name": "watch-experience",
        "hypothesis_template": "Watching this movie makes people feel {}.",
    },
    {
        "name": "recommendation-tone",
        "hypothesis_template": "This film would receive a {} recommendation from viewers.",
    },
]

# Use a small sample for quick prompt comparison
zs2_eval_sample = train_df.sample(n=min(250, len(train_df)), random_state=SEED)

variant_results_zs2 = []

for variant in prompt_variants_zs2:
    preds = []

    for text in zs2_eval_sample["reviews"].fillna("").astype(str).tolist():
        out = zs_classifier_2(
            text,
            candidate_labels=candidate_labels_zs2,
            hypothesis_template=variant["hypothesis_template"],
            multi_label=False,
        )
        top_label_text = out["labels"][0]
        preds.append(label_text_to_id_zs2[top_label_text])

    y_true = zs2_eval_sample["label"].tolist()
    y_pred = preds

    variant_results_zs2.append(
        {
            "variant": variant["name"],
            "hypothesis_template": variant["hypothesis_template"],
            "accuracy": accuracy_score(y_true, y_pred),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
        }
    )

zs2_prompt_results_df = pd.DataFrame(variant_results_zs2).sort_values(
    by=["f1_weighted", "accuracy"], ascending=False
)

print("Prompt variant comparison (Model 2, sample):")
print(zs2_prompt_results_df)

BEST_PROMPT_ZS2 = zs2_prompt_results_df.iloc[0]["hypothesis_template"]
print("\nSelected BEST_PROMPT_ZS2:", BEST_PROMPT_ZS2)

Prompt variant comparison (Model 2, sample):
               variant                                hypothesis_template  \
1     watch-experience          Watching this movie makes people feel {}.   
0     public-reception    The public reception of this movie would be {}.   
2  recommendation-tone  This film would receive a {} recommendation fr...   

   accuracy  f1_weighted  
1     0.264     0.275508  
0     0.220     0.220401  
2     0.200     0.213081  

Selected BEST_PROMPT_ZS2: Watching this movie makes people feel {}.


In [31]:
if "BEST_PROMPT_ZS2" not in globals() or not BEST_PROMPT_ZS2:
    BEST_PROMPT_ZS2 = "The public reception of this movie would be {}."

# Ensure candidate labels/mapping exist even if the previous cell wasn't run.
if "candidate_labels_zs2" not in globals() or "label_text_to_id_zs2" not in globals():
    if len(label_ids) == 5:
        candidate_labels_zs2 = ["very negative", "negative", "neutral", "positive", "very positive"]
        label_text_to_id_zs2 = {
            "very negative": label_ids[0],
            "negative": label_ids[1],
            "neutral": label_ids[2],
            "positive": label_ids[3],
            "very positive": label_ids[4],
        }
    else:
        candidate_labels_zs2 = [str(lbl) for lbl in label_ids]
        label_text_to_id_zs2 = {str(lbl): lbl for lbl in label_ids}

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

def predict_zs2_labels(texts, batch_size=16, desc="Predicting"):
    texts = ["" if pd.isna(t) else str(t) for t in texts]
    preds = []
    iterator = range(0, len(texts), batch_size)
    if tqdm is not None:
        iterator = tqdm(iterator, total=(len(texts) + batch_size - 1) // batch_size, desc=desc)

    for i in iterator:
        batch_texts = texts[i : i + batch_size]
        outputs = zs_classifier_2(
            batch_texts,
            candidate_labels=candidate_labels_zs2,
            hypothesis_template=BEST_PROMPT_ZS2,
            multi_label=False,
            truncation=True,
        )

        if isinstance(outputs, dict):
            outputs = [outputs]

        for out in outputs:
            top_label_text = out["labels"][0]
            preds.append(label_text_to_id_zs2[top_label_text])

    return preds

# Take a validation split from train, and run it like a test split using prelabelled values for metrics
if "val_split_df_m2" in globals() and "text" in val_split_df_m2.columns and "label" in val_split_df_m2.columns:
    zs2_val_df = val_split_df_m2.copy().reset_index(drop=True)
else:
    zs2_val_df = train_df.sample(n=min(1000, len(train_df)), random_state=SEED).copy().reset_index(drop=True)
    zs2_val_df["text"] = zs2_val_df["reviews"].fillna("").astype(str)

zs2_val_preds = predict_zs2_labels(
    zs2_val_df["text"].tolist(),
    batch_size=16,
    desc="Validation zero-shot",
)

zs2_val_true = zs2_val_df["label"].tolist()
zs2_val_acc = accuracy_score(zs2_val_true, zs2_val_preds)
zs2_val_f1w = f1_score(zs2_val_true, zs2_val_preds, average="weighted")
zs2_val_f1m = f1_score(zs2_val_true, zs2_val_preds, average="macro")

print("Zero-shot Model 2 metrics (validation):")
print(f"Accuracy: {zs2_val_acc:.4f}")
print(f"F1 (Weighted): {zs2_val_f1w:.4f}")
print(f"F1 (Macro): {zs2_val_f1m:.4f}")

# Now run the actual unlabelled test split
test_texts_zs2 = test_df["reviews"].fillna("").astype(str).tolist()
zs2_test_preds = predict_zs2_labels(
    test_texts_zs2,
    batch_size=16,
    desc="Test zero-shot",
)

zs2_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": zs2_test_preds,
    }
)

print("\nZero-shot Model 2 test prediction preview:")
print(zs2_test_results.head())
print("Total predictions:", len(zs2_test_results))
print("Label distribution:\n", zs2_test_results["predicted_label"].value_counts().sort_index())

zs2_test_results.to_csv("zs_model_2_test_predictions.csv", index=False)
print("Saved: zs_model_2_test_predictions.csv")

Validation zero-shot:   0%|          | 0/563 [00:00<?, ?it/s]

Zero-shot Model 2 metrics (validation):
Accuracy: 0.2573
F1 (Weighted): 0.3072
F1 (Macro): 0.1590


Test zero-shot:   0%|          | 0/625 [00:00<?, ?it/s]


Zero-shot Model 2 test prediction preview:
      id  predicted_label
0  74233                2
1  28972                2
2  31376                2
3  18117                4
4  72814                2
Total predictions: 10000
Label distribution:
 predicted_label
1     290
2    3935
3    1250
4    4027
5     498
Name: count, dtype: int64
Saved: zs_model_2_test_predictions.csv


### Discussion — Zero-shot Classification (10%)

Answer all of the following:

- Talk about the models used, size, pretraining data, and required compute to pretrain.
- Describe how you prompted the models in order to get it to solve your classification task.
- What other prompts did you consider and why do you think they didn’t work well?

*BART-large-MNLI, consisting of roughly 406M parameters, is a sequence-to-sequence model with a bidirectional encoder and autoregressive decoder. It was pretrained on a large English corpus consisting of roughly 160GB total text. It was then fine-tuned on the Multi-Genre NLI (MNLI) dataset to enable zero-shot classification. The original BART pretraining was conducted on 64 V100 32GB GPUs for roughly 400k steps, amounting to approximately 1–2 weeks of compute. DistilBERT-base-uncased-MNLI, consisting of roughly 66M parameters, is a distilled version of BERT-base, trained on BookCorpus and English Wikipedia using knowledge distillation from BERT-base. It retains ~97% of BERT's performance at 60% of the size. According to the original DistilBERT paper, pretraining used 8 V100 GPUs for approximately 90 hours. This checkpoint was then fine-tuned on MNLI to enable zero-shot classification.*

*Both models use NLI-based zero-shot classification: a hypothesis template is filled with each candidate label, and the model scores how likely that hypothesis is entailed by the input text. I mapped the five rating classes (1–5) to five descriptive phrases: very negative, negative, neutral, positive, and very positive. This gives the models semantically meaningful label names rather than raw integers. For BART-large-MNLI, the three candidates were: "This movie review is {}.", "The overall audience sentiment is {}.", and "The critic's stance toward this movie is {}." For DistilBERT-MNLI, the three candidates were: "The public reception of this movie would be {}.", "Watching this movie makes people feel {}.", and "This film would receive a {} recommendation from viewers."*

*For BART, the templates "The critic's stance toward this movie is {}." and "This movie review is {}." were the non-selected alternatives. I believe the critic-stance framing is too narrow since it focuses on critic opinion specifically, which may not align well with the broader audience-sentiment reviews in the dataset. The direct-sentiment template is very generic and gives the model less contextual signal — this is a weaker hypothesis than one that describes the audience's overall reaction. For DistilBERT, the templates "Watching this movie makes people feel {}." and "This film would receive a {} recommendation from viewers." were the alternatives. The first template conflates emotional affect during viewing with an overall rating, which may cause confusion on neutral or mixed reviews. The second template anchors the prediction on a recommendation signal rather than raw sentiment. Pairing gradations like "very negative" with a recommendation framing is an awkward semantic fit, likely reducing precision at the fine-grained ends of the label scale.*

---
## 1.4 Baselines (#3.3)

### Bag-of-Words Classifier

In [33]:
from sklearn.model_selection import train_test_split

BOW_TEXT_COLS = ["title", "synopsis", "reviews"]

def build_bow_text(df, text_cols):
    text_df = df[text_cols].fillna("").astype(str)
    return text_df.apply(lambda row: " ".join(part.strip() for part in row if part.strip()), axis=1)

X_all_bow = build_bow_text(train_df, BOW_TEXT_COLS)
y_all_bow = train_df["label"].astype(int)
X_test_bow = build_bow_text(test_df, BOW_TEXT_COLS)

# Hold out a validation split so the baseline has measurable metrics
X_train_bow, X_val_bow, y_train_bow, y_val_bow = train_test_split(
    X_all_bow,
    y_all_bow,
    test_size=0.1,
    random_state=SEED,
    stratify=y_all_bow,
)

bow_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000,
    strip_accents="unicode",
)

X_train_bow_features = bow_vectorizer.fit_transform(X_train_bow)
X_val_bow_features = bow_vectorizer.transform(X_val_bow)

bow_classifier = LogisticRegression(
    max_iter=1000,
    multi_class="auto",
    random_state=SEED,
)
bow_classifier.fit(X_train_bow_features, y_train_bow)

bow_val_preds = bow_classifier.predict(X_val_bow_features)
bow_val_acc = accuracy_score(y_val_bow, bow_val_preds)
bow_val_f1w = f1_score(y_val_bow, bow_val_preds, average="weighted")
bow_val_f1m = f1_score(y_val_bow, bow_val_preds, average="macro")

print("BoW validation metrics:")
print(f"Accuracy: {bow_val_acc:.4f}")
print(f"F1 (Weighted): {bow_val_f1w:.4f}")
print(f"F1 (Macro): {bow_val_f1m:.4f}")
print("Vocabulary size:", len(bow_vectorizer.vocabulary_))

# Retrain on the full labeled training set before generating test predictions
X_all_bow_features = bow_vectorizer.fit_transform(X_all_bow)
X_test_bow_features = bow_vectorizer.transform(X_test_bow)
bow_classifier.fit(X_all_bow_features, y_all_bow)

bow_test_preds = bow_classifier.predict(X_test_bow_features)

bow_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": bow_test_preds,
    }
)

print("\nBoW classifier test prediction preview:")
print(bow_test_results.head())
print("Total predictions:", len(bow_test_results))
print("Label distribution:\n", bow_test_results["predicted_label"].value_counts().sort_index())

bow_test_results.to_csv("bow_test_predictions.csv", index=False)
print("Saved: bow_test_predictions.csv")

if "label" in test_df.columns:
    bow_test_acc = accuracy_score(test_df["label"], bow_test_preds)
    bow_test_f1w = f1_score(test_df["label"], bow_test_preds, average="weighted")
    bow_test_f1m = f1_score(test_df["label"], bow_test_preds, average="macro")
    print("\nBoW metrics on test split:")
    print(f"Accuracy: {bow_test_acc:.4f}")
    print(f"F1 (Weighted): {bow_test_f1w:.4f}")
    print(f"F1 (Macro): {bow_test_f1m:.4f}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


BoW validation metrics:
Accuracy: 0.7826
F1 (Weighted): 0.7610
F1 (Macro): 0.3707
Vocabulary size: 50000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



BoW classifier test prediction preview:
      id  predicted_label
0  74233                3
1  28972                3
2  31376                3
3  18117                3
4  72814                3
Total predictions: 10000
Label distribution:
 predicted_label
2     140
3    7826
4    2034
Name: count, dtype: int64
Saved: bow_test_predictions.csv


### Simple Baselines: Majority Class & Random

In [34]:
from sklearn.model_selection import train_test_split

baseline_train_df, baseline_val_df = train_test_split(
    train_df[["label"]].copy(),
    test_size=0.1,
    random_state=SEED,
    stratify=train_df["label"],
)

baseline_train_labels = baseline_train_df["label"].astype(int)
baseline_val_labels = baseline_val_df["label"].astype(int)
majority_label = int(baseline_train_labels.mode().iloc[0])
all_label_values = sorted(train_df["label"].astype(int).unique().tolist())

# Majority class baseline
majority_val_preds = np.full(len(baseline_val_labels), majority_label)
majority_acc = accuracy_score(baseline_val_labels, majority_val_preds)
majority_f1w = f1_score(baseline_val_labels, majority_val_preds, average="weighted")
majority_f1m = f1_score(baseline_val_labels, majority_val_preds, average="macro")

print("Majority class baseline:")
print("Predicted label:", majority_label)
print(f"Accuracy: {majority_acc:.4f}")
print(f"F1 (Weighted): {majority_f1w:.4f}")
print(f"F1 (Macro): {majority_f1m:.4f}")

majority_test_preds = np.full(len(test_df), majority_label)
majority_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": majority_test_preds,
    }
)
majority_test_results.to_csv("majority_baseline_test_predictions.csv", index=False)
print("Saved: majority_baseline_test_predictions.csv")

# Random baseline
rng = np.random.default_rng(SEED)
random_val_preds = rng.choice(all_label_values, size=len(baseline_val_labels), replace=True)
random_acc = accuracy_score(baseline_val_labels, random_val_preds)
random_f1w = f1_score(baseline_val_labels, random_val_preds, average="weighted")
random_f1m = f1_score(baseline_val_labels, random_val_preds, average="macro")

print("\nRandom baseline:")
print(f"Accuracy: {random_acc:.4f}")
print(f"F1 (Weighted): {random_f1w:.4f}")
print(f"F1 (Macro): {random_f1m:.4f}")

random_test_preds = rng.choice(all_label_values, size=len(test_df), replace=True)
random_test_results = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "predicted_label": random_test_preds,
    }
)
random_test_results.to_csv("random_baseline_test_predictions.csv", index=False)
print("Saved: random_baseline_test_predictions.csv")

Majority class baseline:
Predicted label: 3
Accuracy: 0.6448
F1 (Weighted): 0.5055
F1 (Macro): 0.1568
Saved: majority_baseline_test_predictions.csv

Random baseline:
Accuracy: 0.2104
F1 (Weighted): 0.2843
F1 (Macro): 0.1357
Saved: random_baseline_test_predictions.csv


### Discussion — Baselines (5%)

- Describe the setup for your baselines.
- Include enough detail that someone else could reproduce your approach based on reading this section.

*The BoW baseline uses a TF-IDF vectorizer paired with a Logistic Regression classifier. Three text columns from the training data, title, synopsis, and reviews, were concatenated into a single string per example with a space separator. Any missing column values were treated as empty strings. The TF-IDF vectorizer was configured with unigrams and bigrams, a minimum document frequency of 2, a vocabulary cap of 50,000 features, and Unicode accent stripping. To get validation metrics, 10% of the training data was held out as a stratified validation split. The vectorizer was fit on the 90% training portion and then used to transform the validation portion. After validation metrics were recorded, the vectorizer and classifier were retrained on the full labeled training set before generating test predictions, ensuring the final model had seen as much data as possible.*

*The majority class baseline always predicts the single most frequent label in the training data. The training set was split 90/10 to produce a validation set. The majority label was computed as the mode of the 90% training portion. All validation and test examples received this label as their prediction. This represents the weakest meaningful baseline, since it requires no model at all and simply reflects the class imbalance in the data. The random baseline samples a label uniformly at random from the set of all possible label values. Predictions are drawn using numpy to ensure reproducibility. Like the majority baseline, it uses the same stratified 90/10 validation split for evaluation. Random predictions will on average perform at the level of a uniform random guesser, serving as the absolute lower bound for any meaningful model.*

---
## 1.5 Results Summary

In [35]:
required_metric_vars = [
    "zs1_val_acc", "zs1_val_f1m", "zs1_val_f1w",
    "zs2_val_acc", "zs2_val_f1m", "zs2_val_f1w",
    "bow_val_acc", "bow_val_f1m", "bow_val_f1w",
    "majority_acc", "majority_f1m", "majority_f1w",
    "random_acc", "random_f1m", "random_f1w",
]
missing_metric_vars = [name for name in required_metric_vars if name not in globals()]
if missing_metric_vars:
    raise ValueError(
        "Run the zero-shot and baseline evaluation cells before creating the results table. Missing: "
        + ", ".join(missing_metric_vars)
    )

def compute_finetuned_validation_metrics(trainer, validation_dataset):
    predictions = trainer.predict(validation_dataset)
    pred_ids = np.argmax(predictions.predictions, axis=-1)
    true_ids = predictions.label_ids
    return {
        "accuracy": accuracy_score(true_ids, pred_ids),
        "f1_macro": f1_score(true_ids, pred_ids, average="macro"),
        "f1_weighted": f1_score(true_ids, pred_ids, average="weighted"),
    }

ft_model_1_metrics = compute_finetuned_validation_metrics(trainer_1, val_for_trainer_1)
ft_model_2_metrics = compute_finetuned_validation_metrics(trainer_2, val_for_trainer_2)

results = {
    "Model": [
        f"Fine-tuned: {MODEL_1_NAME}",
        f"Fine-tuned: {MODEL_2_NAME}",
        f"Zero-shot: {ZS_MODEL_1_NAME}",
        f"Zero-shot: {ZS_MODEL_2_NAME}",
        "BoW Classifier",
        "Majority Class Baseline",
        "Random Baseline",
    ],
    "Accuracy": [
        ft_model_1_metrics["accuracy"],
        ft_model_2_metrics["accuracy"],
        zs1_val_acc,
        zs2_val_acc,
        bow_val_acc,
        majority_acc,
        random_acc,
    ],
    "F1 (Macro)": [
        ft_model_1_metrics["f1_macro"],
        ft_model_2_metrics["f1_macro"],
        zs1_val_f1m,
        zs2_val_f1m,
        bow_val_f1m,
        majority_f1m,
        random_f1m,
    ],
    "F1 (Weighted)": [
        ft_model_1_metrics["f1_weighted"],
        ft_model_2_metrics["f1_weighted"],
        zs1_val_f1w,
        zs2_val_f1w,
        bow_val_f1w,
        majority_f1w,
        random_f1w,
    ],
}

results_df = pd.DataFrame(results)
results_df[["Accuracy", "F1 (Macro)", "F1 (Weighted)"]] = results_df[["Accuracy", "F1 (Macro)", "F1 (Weighted)"]].round(4)
results_df

,Model,Accuracy,F1 (Macro),F1 (Weighted)
0,Fine-tuned: distilroberta-base,0.6448,0.1568,0.5055
1,Fine-tuned: prajjwal1/bert-tiny,0.7192,0.3141,0.6979
2,Zero-shot: facebook/bart-large-mnli,0.3436,0.1945,0.3795
3,Zero-shot: typeform/distilbert-base-uncased-mnli,0.2573,0.1590,0.3072
4,BoW Classifier,0.7826,0.3707,0.7610
5,Majority Class Baseline,0.6448,0.1568,0.5055
6,Random Baseline,0.2104,0.1357,0.2843


### Discussion — Results (15%)

Write a substantive analysis of your results.

- Show a table that includes all of your results. In the end, you should have 7 rows. Two fine-tuned models, two zero-shot models, one BoW approach, and two simple baselines (random and majority class). Include one column for each evaluation metric that you computed.

- Describe what observations and analysis you have from these results. Do not just repeat the information that readers can directly find in your table.

- Think about what these results mean. For example, if you had to make recommendations to someone else who wanted to build a model for this task, what would you say to them that would be important for them to consider?

*One thing we can observe is that our first fine-tuned model performs exactly the same as the majority class baseline, this is a surprising result that we managed to obtain the exact same results as the majority class baseline considering we specifically fine-tuned the first model. One observation we can make from this is that fine-tuning with certain parameters will affect the outcome of the models performance. Considering the size of the model's parameters and the amount of data it was trained on, I had to tweak some parameters so that training the model wouldn't take more than half an hour even on T4 GPUs, otherwise training times could have been as high as 3 hours. Immediately from this we can conclude that modifying certain parameters increases the time it takes to train or fine-tune a model. Also we can note the fact that the BoW Classifier outperformed both fine-tuned models. Since we were given the freedom to select our chosen models and I randomly selected the 2 that I had, we can conclude that the model's performance on a task depends on the choice of pre-trained model as well. Lastly, we can generally conclude that the random baseline holds as the worst performing model with all metrics considered, which is how it should be performing. Next to the random baseline, we see both zero-shot models having the worst performances which is expected.*

*Based on these results, the most important thing I would recommend for people to consider when building a model for this task is for them to make sure they have enough compute time to allow the models to train with proper training parameters to obtain better training metrics. Or if you don't have enough time like I did, carefully modify training parameters one by one to mitigate your model's performance loss but on the other hand save in training time. In terms of which models to use themselves for the task, based on the results I would recommend fine-tuning an existing pretrained model, or to explore the BoW Classifier to maximize model performance.*

---
# PART 2 — Self-Chosen Secondary Task

For this task, you only need to complete **either** #3.1 (fine-tuning) **or** #3.2 (zero-shot) — not both, and not the baselines.  

**Secondary task chosen:** *(name it here)*

## 2.1 Secondary Task Dataset

In [ ]:
# Loading and exploring dataset
from pathlib import Path
import pandas as pd

DATA_DIR_2 = Path("/content/drive/MyDrive/Start_kit/")

train_df_2 = pd.read_csv(DATA_DIR_2 / "train.csv")
val_df_2 = pd.read_csv(DATA_DIR_2 / "val.csv")
test_df_2 = pd.read_csv(DATA_DIR_2 / "test_input.csv")

print("Train shape:", train_df_2.shape)
print("Validation shape:", val_df_2.shape)
print("Test shape:", test_df_2.shape)

def explore_split(df, split_name):
    print(f"\n=== {split_name} ===")
    print("Columns:", list(df.columns))
    print("Dtypes:\n", df.dtypes)
    print("\nMissing values:\n", df.isnull().sum())

    if "Label" in df.columns:
        print("\nLabel distribution:")
        print(df["Label"].value_counts().sort_index())

    print("\nSample rows:")
    display(df.head(3))


explore_split(train_df_2, "Train")
explore_split(val_df_2, "Validation")
explore_split(test_df_2, "Test")

Train shape: (720, 4)
Validation shape: (240, 4)
Test shape: (240, 3)

=== Train ===
Columns: ['ID', 'Post', 'Comment', 'Label']
Dtypes:
 ID         float64
Post        object
Comment     object
Label       object
dtype: object

Missing values:
 ID         0
Post       0
Comment    0
Label      0
dtype: int64

Label distribution:
Label
direct       492
off-topic     56
related      172
Name: count, dtype: int64

Sample rows:


,ID,Post,Comment,Label
0,163.0,"""Is there a politician you strongly disagree w...","""Mamdani. Far too left for me but good persona...",direct
1,146.0,"""How would you feel about Askreddit banning ""H...","""Yes please. It's getting flooded with off-top...",direct
2,629.0,"""Best sports stream site now? Since stream Eas...","""dude was preparing for the Pereira fight""",off-topic



=== Validation ===
Columns: ['ID', 'Post', 'Comment', 'Label']
Dtypes:
 ID         float64
Post        object
Comment     object
Label       object
dtype: object

Missing values:
 ID         0
Post       0
Comment    0
Label      0
dtype: int64

Label distribution:
Label
direct       164
off-topic     18
related       58
Name: count, dtype: int64

Sample rows:


,ID,Post,Comment,Label
0,28.0,"""People who swore an oath to defend the consti...","""I'm a veteran and father of 2 mixed teenagers...",direct
1,1144.0,"""This teacher of ours lectured us on how diffi...","""Well, when I was learning coding (it was C++)...",related
2,21.0,"""People who swore an oath to defend the consti...","""I am disgusted. I served on active duty for o...",direct



=== Test ===
Columns: ['ID', 'Post', 'Comment']
Dtypes:
 ID         float64
Post        object
Comment     object
dtype: object

Missing values:
 ID         1
Post       0
Comment    0
dtype: int64

Sample rows:


,ID,Post,Comment
0,451.0,"""Is the Tv show ""Friends"" funny?""","""Joey always makes me laugh. Chandler is often..."
1,190.0,"""What is the best political system?""","""Anarchy, prove me wrong. Anarchy gives everyo..."
2,444.0,"""Important question: Was Friends actually funny?""","""People seemed to like it. Personally, I never..."


## 2.2 Approach — Fine-tuning OR Zero-shot

*(Complete only one of the two sections below. Delete the one you are not doing.)*

### Option A: Fine-tuning (#3.1)
*(Complete this section if you chose to fine-tune models on the secondary task)*

In [ ]:
# Loading new pretrained model to fine-tune
MODEL_3_NAME = "distilbert-base-uncased"

num_labels_3 = train_df_2["Label"].nunique()
label_ids_3 = sorted(train_df_2["Label"].unique())
id2label_3 = {i: str(lbl) for i, lbl in enumerate(label_ids_3)}
label2id_3 = {str(lbl): i for i, lbl in enumerate(label_ids_3)}

tokenizer_3 = AutoTokenizer.from_pretrained(MODEL_3_NAME)
model_3 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_3_NAME,
    num_labels=num_labels_3,
)

model_3.config.id2label = id2label_3
model_3.config.label2id = label2id_3

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Tokenization

from datasets import Dataset, DatasetDict

def build_input_text_model_3(df):
    return (
        df[TEXT_COLS_3]
        .fillna("")
        .astype(str)
        .apply(lambda row: " </s> ".join(part.strip() for part in row if part.strip()), axis=1)
    )

train_df_3 = train_df_2.copy()
val_df_3 = val_df_2.copy()
test_df_3 = test_df_2.copy()

train_df_3["text"] = build_input_text_model_3(train_df_3)
val_df_3["text"] = build_input_text_model_3(val_df_3)
test_df_3["text"] = build_input_text_model_3(test_df_3)

test_ids_3 = test_df_3[ID_COL_3].values

raw_ds_3 = DatasetDict(
    {
        "train": Dataset.from_pandas(
            train_df_3[["text", LABEL_COL_3]].rename(columns={LABEL_COL_3: "label"}).reset_index(drop=True),
            preserve_index=False,
        ),
        "validation": Dataset.from_pandas(
            val_df_3[["text", LABEL_COL_3]].rename(columns={LABEL_COL_3: "label"}).reset_index(drop=True),
            preserve_index=False,
        ),
        "test": Dataset.from_pandas(
            test_df_3[["text"]].reset_index(drop=True),
            preserve_index=False,
        ),
    }
)

def tokenize_batch_model_3(batch):
    return tokenizer_3(batch["text"], truncation=True, padding="max_length", max_length=256)

tokenized_ds_3 = raw_ds_3.map(tokenize_batch_model_3, batched=True)

tokenized_train_3 = tokenized_ds_3["train"].remove_columns(["text"])
tokenized_val_3 = tokenized_ds_3["validation"].remove_columns(["text"])
tokenized_test_3 = tokenized_ds_3["test"].remove_columns(["text"])

print(tokenized_train_3)
print(tokenized_val_3)
print(tokenized_test_3)

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Using label column: Label
Using text columns: ['Post', 'Comment']
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 720
})
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 240
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 240
})


In [ ]:
from transformers import DataCollatorWithPadding

data_collator_3 = DataCollatorWithPadding(tokenizer=tokenizer_3)

def compute_metrics_model_3(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_weighted": f1_score(labels, preds, average="weighted"),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

def add_trainer_labels_model_3(batch):
    return {"labels": [label2id_3[str(lbl)] for lbl in batch["label"]]}

train_for_trainer_3 = tokenized_train_3.map(add_trainer_labels_model_3, batched=True)
val_for_trainer_3 = tokenized_val_3.map(add_trainer_labels_model_3, batched=True)

train_for_trainer_3 = train_for_trainer_3.remove_columns(["label"])
val_for_trainer_3 = val_for_trainer_3.remove_columns(["label"])

train_for_trainer_3.set_format("torch")
val_for_trainer_3.set_format("torch")
tokenized_test_3.set_format("torch")

training_args_3 = TrainingArguments(
    output_dir="./outputs/model_3_distilbert_secondary",
    num_train_epochs=3,
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    seed=SEED,
)

trainer_3 = Trainer(
    model=model_3,
    args=training_args_3,
    train_dataset=train_for_trainer_3,
    eval_dataset=val_for_trainer_3,
    data_collator=data_collator_3,
    compute_metrics=compute_metrics_model_3,
)

train_result_3 = trainer_3.train()

print("Selected learning_rate:", training_args_3.learning_rate)
print("Selected weight_decay:", training_args_3.weight_decay)
print("Train loss:", train_result_3.training_loss)

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.765422,0.759666,0.679167,0.611504,0.356098
2,0.739624,0.798201,0.633333,0.605689,0.372798
3,0.593474,0.863184,0.637500,0.610559,0.378170


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Selected learning_rate: 3e-05
Selected weight_decay: 0.01
Train loss: 0.6772188186645508


In [ ]:
# Same as before, our new data set also has testing data that isn't prelabelled
test_predictions_3 = trainer_3.predict(tokenized_test_3)
test_pred_class_ids_3 = np.argmax(test_predictions_3.predictions, axis=-1)

predicted_labels_3 = [id2label_3[idx] for idx in test_pred_class_ids_3]

model_3_test_results = pd.DataFrame(
    {
        "ID": test_ids_3,
        "predicted_label": predicted_labels_3,
    }
)

print(model_3_test_results.head())
print("Total predictions:", len(model_3_test_results))
print("Label distribution:\n", model_3_test_results["predicted_label"].value_counts().sort_index())

model_3_test_results.to_csv("model_3_test_predictions.csv", index=False)
print("Saved: model_3_test_predictions.csv")

      ID predicted_label
0  451.0          direct
1  190.0          direct
2  444.0          direct
3  443.0          direct
4  878.0          direct
Total predictions: 240
Label distribution:
 predicted_label
direct     222
related     18
Name: count, dtype: int64
Saved: model_3_test_predictions.csv


## 2.3 Secondary Task Results

In [ ]:
def compute_model_3_validation_metrics(trainer, validation_dataset):
    predictions = trainer.predict(validation_dataset)
    pred_ids = np.argmax(predictions.predictions, axis=-1)
    true_ids = predictions.label_ids
    return {
        "accuracy": accuracy_score(true_ids, pred_ids),
        "f1_macro": f1_score(true_ids, pred_ids, average="macro"),
        "f1_weighted": f1_score(true_ids, pred_ids, average="weighted"),
    }

model_3_metrics = compute_model_3_validation_metrics(trainer_3, val_for_trainer_3)

secondary_results = {
    "Model": [f"Fine-tuned: {MODEL_3_NAME}"],
    "Accuracy": [model_3_metrics["accuracy"]],
    "F1 (Macro)": [model_3_metrics["f1_macro"]],
    "F1 (Weighted)": [model_3_metrics["f1_weighted"]],
}

secondary_results_df = pd.DataFrame(secondary_results)
secondary_results_df[["Accuracy", "F1 (Macro)", "F1 (Weighted)"]] = secondary_results_df[["Accuracy", "F1 (Macro)", "F1 (Weighted)"]].round(4)

print("Secondary Task — Model 3 Validation Metrics:")
print(secondary_results_df)
print("\nDetailed metrics:")
print(f"Accuracy: {model_3_metrics['accuracy']:.4f}")
print(f"F1 (Macro): {model_3_metrics['f1_macro']:.4f}")
print(f"F1 (Weighted): {model_3_metrics['f1_weighted']:.4f}")

Secondary Task — Model 3 Validation Metrics:
                                 Model  Accuracy  F1 (Macro)  F1 (Weighted)
0  Fine-tuned: distilbert-base-uncased    0.6792      0.3561         0.6115

Detailed metrics:
Accuracy: 0.6792
F1 (Macro): 0.3561
F1 (Weighted): 0.6115


### Discussion — Secondary Task (10%)

1. Describe the steps you took for the second task that you chose.
2. Did you fine-tune models, perform zero-shot classification, or both?
3. How did these models perform compared to the baselines provided?

*For the secondary task, I followed essentially the same end-to-end fine-tuning workflow as Part 1, but adapted it to the new dataset schema. First, I loaded all the dataset splits, inspected column types and missing values, and verified label distribution on the labeled splits. Then I selected distilbert-base-uncased as my third pre-trained model and built a single model input string by concatenating the available text fields with a separator so the model could jointly use all text evidence. I tokenized with truncation and fixed maximum length, converted predefined labels to contiguous IDs and prepared Hugging Face DataSet objects for train/validation/test. For optimization, I fine-tuned the model for 3 epochs using a learning rate of 3e-5, a batch size of 8 for training, and weight decay 0.01, all while tracking validation metrics and loading the best checkpoint at the end. After training, I generated predictions for the unlabeled test split and exported them to an output .csv file, and I computed validation metrics (accuracy, macro-F1, weighted-F1) for reporting.*

*I chose fine-tuning only for this secondary task. The main reason is that this dataset already provided supervised labels in train/validation, so full task adaptation is likely to be more effective than prompt-based transfer. Fine-tuning also lets the model learn task-specific decision boundaries directly from in-domain examples.*

*The fine-tuned model achieved Accuracy = 0.6792, F1 (Macro) = 0.3561, and F1 (Weighted) = 0.6115. Relative to provided baselines, this indicates clear practical improvement overall: accuracy and weighted-F1 suggest the model is capturing the dominant class patterns much better than naive strategies, while macro-F1 is noticeably lower, which signals that minority classes are still harder and that performance is uneven across labels. In other words, the model is meaningfully stronger than baseline behavior, but it has not fully solved class-balance sensitivity.*

---
## Reflection (10%)

Answer all of the following thoughtfully:

1. What did you learn during the completion of this assignment?
2. What was unexpected, if anything?
3. What challenges did you face and how did you overcome them?

*I learned that evaluation metrics can tell very different stories: accuracy and weighted F1 can look strong while macro F1 reveals weaker minority-class behavior. I also learned how much careful preprocessing and label mapping matter in NLP pipelines, especially when combining multiple text fields and moving between fine-tuning and zero-shot setups.*

*One unexpected result was how competitive simpler approaches could be in some settings, and how sensitive transformer performance was to training choices (for example learning rate, batch size, and number of epochs). I expected larger pretrained models to always dominate, but this assignment showed that task fit and tuning details can outweigh raw model size.*

*The main challenge was to do with the long training/evaluation cycles. There wasn't really a good way around this, other than sacrificing the models performance to minimize training time and validation time.*

---
## Code Attribution

List any code snippets, tutorials, or resources you referenced below. Include URLs and a short description of what was adapted.

| Source | URL | What was adapted |
|--------|-----|------------------|
| HuggingFace Trainer tutorial | https://huggingface.co/docs/transformers/training | Fine-tuning loop scaffold |
| *(add more rows as needed)* | | |